# Dataset Groups

This notebook groups [GIFT-Eval](https://github.com/SalesforceAIResearch/gift-eval) datasets based on the models proposed in the [Ensemble TEMPO slides](https://docs.google.com/presentation/d/1GxL_qUQKizv5C_RPzxYiJjxJSPD6-81x1VPOxxYXAl0/edit?slide=id.g35d6bf22e47_0_10#slide=id.g35d6bf22e47_0_10).
- First, we'll group pretraining datasets
- Then, we'll group train-test datasets

Ensemble TEMPO will have models at three different granularities:
- **General** 
  - Three models across three forecasting terms (short, medium, and long)
- **Domain-specific** 
  - 15 models across seven domains and three forecasting terms
- **Dataset-specific** 
  - 97+ models across each dataset
  - **Note:**: Dataset-specific models will only be trained on the train-test split

## Pretraining Split

Load the pretraining split's metadata.

In [31]:
import pandas as pd
from pathlib import Path

split = "pretrain"
metadata_path = Path("resources") / split / "metadata.csv"

df = pd.read_csv(metadata_path)

print(f"Number of unique {split} datasets: {df['name'].nunique()}")
print(f"Number of unique {split} terms: {df['term'].nunique()}")
print(f"Total number of {split} datasets: {len(df)}")
df.head()

Number of unique pretrain datasets: 152
Number of unique pretrain terms: 3
Total number of pretrain datasets: 456


,name,term,freq,domain,num_series,target_dim,_min_series_length,sum_series_length,prediction_length,windows
0,bull,short,H,Energy,41.0,1,17544,719304,48,20
1,bull,medium,H,Energy,41.0,1,17544,719304,480,4
2,bull,long,H,Energy,41.0,1,17544,719304,720,3
3,cmip6_1885,short,6H,Climate,434176.0,53,7300,3169484800,48,16
4,cmip6_1885,medium,6H,Climate,434176.0,53,7300,3169484800,480,2


Create a DataFrame for the pretraining split's dataset groups.
- Each row represents a different dataset group:
  - `columns` specifies the columns that'll be used for group
  - `values` specifies the unique values in the grouped columns

In [32]:
groups_df = pd.DataFrame(columns=["columns", "values"])
groups_df

,columns,values


### General

View all of the unique forecasting terms and count the number of datasets that belong to each term. Each forecasting term multiplies the original prediciton length by a given multipler:
- "long" multiplies the original prediction length by 15
- "medium" multiplies the original prediction length by 10  
- "short" multiplies the original prediction length by 1 (no change)

In [33]:
def get_count_df(
    df: pd.DataFrame,
    columns: list[str],
    ascending: bool = False,
) -> pd.DataFrame:
    """
    Counts the number of unique combinations of all unique values in the
    specified columns of the given a DataFrame.

    Args:
        df (pd.DataFrame): The input DataFrame.
        columns (list[str]): A list of column names to group by.
        ascending (bool, optional): If True, sort the result in ascending order
            of count. Defaults to False (descending order).

    Returns:
        pd.DataFrame: A DataFrame with the grouped columns and a 'count'
            column, sorted by count.
    """
    count_df = df.groupby(columns).size().reset_index(name="count")
    return count_df.sort_values(by="count", ascending=ascending).reset_index(drop=True)


term_counts = get_count_df(df, columns=["term"], ascending=False)

print(f"Number of unique terms: {len(term_counts)}")
display(term_counts)

Number of unique terms: 3


,term,count
0,long,152
1,medium,152
2,short,152


Create the general models' dataset groups by using all of the datasets that belong to each forecasting term.

In [34]:
for term in df["term"].unique():
    groups_df.loc[len(groups_df)] = ["term", term]

display(groups_df)

,columns,values
0,term,short
1,term,medium
2,term,long


### Domain-Specific

View all of the unique domain-term combinations and count the number of datasets that belong to each domain-term combination.

In [35]:
domain_term_counts = get_count_df(
    df,
    columns=["domain", "term"],
    ascending=False,
)

# Convert each forecasting term into its own column
pivoted_df = domain_term_counts.pivot(
    index="domain",
    columns="term",
    values="count",
).reset_index()

# Remove old "term" column
pivoted_df.columns.name = None

# Reorder columns
pivoted_df = pivoted_df[
    [
        "domain",
        "short",
        "medium",
        "long",
    ]
]

print(f"Number of unique domain-term combinations: {len(domain_term_counts)}")
display(pivoted_df)

Number of unique domain-term combinations: 27


,domain,short,medium,long
0,Climate,67,67,67
1,CloudOps,3,3,3
2,Econ/Fin,17,17,17
3,Energy,28,28,28
4,Healthcare,3,3,3
5,Nature,3,3,3
6,Sales,3,3,3
7,Transport,25,25,25
8,Web,3,3,3


Create the domain-specific models' dataset groups by using all of the datasets that belong to each domain-term combination.
- Semicolons (;) are used to deliminate values corresponding to different columns
- Commas (,) are used to deliminate multiple values that correspond to the same column

In [ ]:
for domain, term in df[["domain", "term"]].drop_duplicates().values:
    groups_df.loc[len(groups_df)] = ["domain;term", f"{domain};{term}"]

for term in df["term"].unique():
    # Add a combined "Climate" and "Nature" group
    groups_df.loc[len(groups_df)] = ["domain;term", f"Climate,Nature;{term}"]

    # Add a combined "Web" and "CloudOps" group
    groups_df.loc[len(groups_df)] = ["domain;term", f"Web,CloudOps;{term}"]

# TODO: Remove medium and long groups for the following domains:
# - Econ/Fin
# - Healthcare
# - Sales
# These domains only have short datasets in the train-test split

groups_df.tail(10)

,columns,values
26,domain;term,Sales;long
27,domain;term,Nature;short
28,domain;term,Nature;medium
29,domain;term,Nature;long
30,domain;term,"Climate,Nature;short"
31,domain;term,"Web,CloudOps;short"
32,domain;term,"Climate,Nature;medium"
33,domain;term,"Web,CloudOps;medium"
34,domain;term,"Climate,Nature;long"
35,domain;term,"Web,CloudOps;long"


View the final dataset groups and save the groups to a CSV file.

In [37]:
output_path = Path("resources") / split / "dataset_configs.csv"
groups_df.to_csv(output_path, index=False)
print(f"Saved dataset configurations to {output_path}")

display(groups_df)

Saved dataset configurations to resources\pretrain\dataset_configs.csv


,columns,values
0,term,short
1,term,medium
2,term,long
3,domain;term,Energy;short
4,domain;term,Energy;medium
5,domain;term,Energy;long
6,domain;term,Climate;short
7,domain;term,Climate;medium
8,domain;term,Climate;long
9,domain;term,Transport;short


## Train-Test Split

Load the train-test split's metadata.

In [38]:
split = "train_test"
metadata_path = Path("resources") / split / "metadata.csv"

df = pd.read_csv(metadata_path)

print(f"Total number of {split} datasets: {len(df)}")
df.head()

Total number of train_test datasets: 97


,name,term,freq,domain,num_series,target_dim,_min_series_length,sum_series_length,prediction_length,windows
0,LOOP_SEATTLE/5T,short,5T,Transport,323,1,105120,33953760,48,20
1,LOOP_SEATTLE/D,short,D,Transport,323,1,365,117895,30,2
2,LOOP_SEATTLE/H,short,H,Transport,323,1,8760,2829480,48,19
3,M_DENSE/D,short,D,Transport,30,1,730,21900,30,3
4,M_DENSE/H,short,H,Transport,30,1,17520,525600,48,20


Create a DataFrame for the pretraining split's dataset groups.
- Each row represents a different dataset group:
  - `columns` specifies the columns that'll be used for group
  - `values` specifies the unique values in the grouped columns

In [39]:
groups_df = pd.DataFrame(columns=["columns", "values"])
groups_df

,columns,values


### General

View all of the unique forecasting terms and count the number of datasets that belong to each term. Each forecasting term multiplies the original prediciton length by a given multipler:
- "long" multiplies the original prediction length by 15
- "medium" multiplies the original prediction length by 10  
- "short" multiplies the original prediction length by 1 (no change)

In [40]:
term_counts = get_count_df(df, columns=["term"], ascending=False)

print(f"Number of unique terms: {len(term_counts)}")
display(term_counts)

Number of unique terms: 3


,term,count
0,short,55
1,long,21
2,medium,21


Create the general models' dataset groups by using all of the datasets that belong to each forecasting term.

In [41]:
for term in df["term"].unique():
    groups_df.loc[len(groups_df)] = ["term", term]

display(groups_df)

,columns,values
0,term,short
1,term,medium
2,term,long


### Domain-Specific

View all of the unique domain-term combinations and count the number of datasets that belong to each domain-term combination.
- **Note:** Econ/Fin, Healthcare, and Sales only have short term datasets

In [42]:
domain_term_counts = get_count_df(
    df,
    columns=["domain", "term"],
    ascending=False,
)

# Convert each forecasting term into its own column
pivoted_df = (
    domain_term_counts.pivot(
        index="domain",
        columns="term",
        values="count",
    )
    .fillna(0)
    .reset_index()
)

# Remove old "term" column
pivoted_df.columns.name = None

# Reorder columns
pivoted_df = pivoted_df[
    [
        "domain",
        "short",
        "medium",
        "long",
    ]
]

print(f"Number of unique domain-term combinations: {len(domain_term_counts)}")
display(pivoted_df)

Number of unique domain-term combinations: 15


,domain,short,medium,long
0,Econ/Fin,6.0,0.0,0.0
1,Energy,16.0,8.0,8.0
2,Healthcare,5.0,0.0,0.0
3,Nature,9.0,3.0,3.0
4,Sales,4.0,0.0,0.0
5,Transport,7.0,4.0,4.0
6,Web/CloudOps,8.0,6.0,6.0


Create the domain-specific models' dataset groups by using all of the datasets that belong to each domain-term combination.

In [43]:
for domain, term in df[["domain", "term"]].drop_duplicates().values:
    groups_df.loc[len(groups_df)] = ["domain,term", f"{domain},{term}"]

display(groups_df)

,columns,values
0,term,short
1,term,medium
2,term,long
3,"domain,term","Transport,short"
4,"domain,term","Web/CloudOps,short"
5,"domain,term","Sales,short"
6,"domain,term","Healthcare,short"
7,"domain,term","Energy,short"
8,"domain,term","Nature,short"
9,"domain,term","Econ/Fin,short"


### Dataset-Specific 

View some of the dataset name-term combinations. 
- **Note:** for the sake of brevity, we'll only display a few of the dataset name-term combinations

In [44]:
name_term_combinations = df[["name", "term"]]

print(f"Number of unique name-term pairs: {len(name_term_combinations)}")
name_term_combinations.head()

Number of unique name-term pairs: 97


,name,term
0,LOOP_SEATTLE/5T,short
1,LOOP_SEATTLE/D,short
2,LOOP_SEATTLE/H,short
3,M_DENSE/D,short
4,M_DENSE/H,short


Create the dataset-specific models' dataset groups by creating a new group for each dataset name-term combination.

In [45]:
for name, term in df[["name", "term"]].drop_duplicates().values:
    groups_df.loc[len(groups_df)] = ["name,term", f"{name},{term}"]

groups_df.tail()

,columns,values
110,"name,term","jena_weather/10T,long"
111,"name,term","jena_weather/H,long"
112,"name,term","kdd_cup_2018_with_missing/H,long"
113,"name,term","solar/10T,long"
114,"name,term","solar/H,long"


View some of the final dataset groups and save the groups to a CSV file.

In [46]:
output_path = Path("resources") / split / "dataset_configs.csv"
groups_df.to_csv(output_path, index=False)
print(f"Saved dataset configurations to {output_path}")

groups_df.head()

Saved dataset configurations to resources\train_test\dataset_configs.csv


,columns,values
0,term,short
1,term,medium
2,term,long
3,"domain,term","Transport,short"
4,"domain,term","Web/CloudOps,short"
